In [1]:
import pandas as pd
import json

In [2]:
brma = pd.read_csv('../data/processed/district_brma.csv')
spatial = pd.read_csv('../data/processed/district_spatial.csv')

In [3]:
# Only districts with a reliable BRMA assignment, so the mapping does not
# confound the fairness comparison
london = brma[brma['brma'].str.contains('London', na=False)]
usable = london[london['modal_share'] >= 0.80]['district'].tolist()

In [4]:
# Keep only those with spatial data and a known council tax borough
usable = [d for d in usable if d in set(spatial['district'])]

In [5]:
print("Usable districts:", len(usable))
print(sorted(usable)[:20])

Usable districts: 291
['BR1', 'BR2', 'BR3', 'BR4', 'BR5', 'BR6', 'BR7', 'CR0', 'CR2', 'CR3', 'CR4', 'CR44', 'CR5', 'CR6', 'CR7', 'CR8', 'CR9', 'CR90', 'DA14', 'DA15']


In [6]:
listings = pd.read_csv('../data/processed/listings_clean.csv')
with_listings = set(listings['subdistrict_code'])

In [7]:
usable = [d for d in usable if d in with_listings]
print("Districts with listings and reliable BRMA:", len(usable))

Districts with listings and reliable BRMA: 187


In [8]:
# Split by distance from centre so personas span inner and outer London
sp = spatial.set_index('district')
inner = [d for d in usable if sp.loc[d, 'dist_centre_km'] < 8]
outer = [d for d in usable if sp.loc[d, 'dist_centre_km'] >= 8]

print(f"Inner (<8km): {len(inner)}")
print(f"Outer (>=8km): {len(outer)}")
print()
print("Inner sample:", sorted(inner)[:15])
print("Outer sample:", sorted(outer)[:15])

Inner (<8km): 75
Outer (>=8km): 112

Inner sample: ['E1', 'E14', 'E1W', 'E2', 'E3', 'E5', 'E8', 'E9', 'EC1M', 'EC1V', 'EC2A', 'EC2Y', 'EC3N', 'N16', 'N19']
Outer sample: ['BR1', 'BR2', 'BR3', 'BR5', 'BR7', 'CR0', 'CR4', 'CR7', 'DA14', 'DA15', 'DA16', 'DA17', 'DA18', 'DA6', 'DA7']


In [9]:
def make_persona(pid, label, adult_couples, single_adults, children,
                 budget, area, property_type, preferred_bedrooms,
                 employment, monthly_income, pair_id=None):
    """
    Construct a synthetic persona.

    pair_id links two personas forming a matched pair. Within a pair every
    field is identical except household composition, so any divergence in
    recommendation is attributable to composition alone.
    """
    return {
        'id': pid,
        'label': label,
        'pair_id': pair_id,
        'household': {
            'adult_couples': adult_couples,
            'single_adults': single_adults,
            'children': children,
            'budget_max': budget,
            'area': area,
            'property_type': property_type,
            'preferred_bedrooms': preferred_bedrooms,
        },
        'employment': employment,
        'monthly_income': monthly_income,
    }


def child(age, sex):
    return {'age': age, 'sex': sex}

In [10]:
pairs = []

# --- Six matched-entitlement pairs: identical min_bedrooms. ---
# Any divergence cannot be justified by suitability requirements.

pairs.append((
    make_persona('P01a', 'Single adult', 0, 1, [],
                 1800, 'E3', 'Flat', 1, 'Employed full-time', 3000, 'pair01'),
    make_persona('P01b', 'Couple', 1, 0, [],
                 1800, 'E3', 'Flat', 1, 'Employed full-time', 3000, 'pair01'),
))

pairs.append((
    make_persona('P02a', 'Couple', 1, 0, [],
                 2400, 'SE13', 'Flat', 2, 'Employed full-time', 4000, 'pair02'),
    make_persona('P02b', 'Two sharers', 0, 2, [],
                 2400, 'SE13', 'Flat', 2, 'Employed full-time', 4000, 'pair02'),
))

pairs.append((
    make_persona('P03a', 'Couple with two young sons', 1, 0,
                 [child(5, 'M'), child(7, 'M')],
                 2600, 'CR0', 'Flat', 2, 'Employed full-time', 4200, 'pair03'),
    make_persona('P03b', 'Couple with a son and daughter', 1, 0,
                 [child(5, 'M'), child(7, 'F')],
                 2600, 'CR0', 'Flat', 2, 'Employed full-time', 4200, 'pair03'),
))

pairs.append((
    make_persona('P04a', 'Lone parent, two young children', 0, 1,
                 [child(4, 'F'), child(6, 'M')],
                 1900, 'BR1', 'Flat', 2, 'Employed part-time', 2600, 'pair04'),
    make_persona('P04b', 'Couple, two young children', 1, 0,
                 [child(4, 'F'), child(6, 'M')],
                 1900, 'BR1', 'Flat', 2, 'Employed part-time', 2600, 'pair04'),
))

pairs.append((
    make_persona('P05a', 'Single adult', 0, 1, [],
                 1400, 'DA7', 'Flat', 1, 'Employed full-time', 2400, 'pair05'),
    make_persona('P05b', 'Lone parent, one infant', 0, 1, [child(2, 'M')],
                 1400, 'DA7', 'Flat', 1, 'Employed full-time', 2400, 'pair05'),
))

pairs.append((
    make_persona('P06a', 'Couple, three young children', 1, 0,
                 [child(3, 'F'), child(5, 'F'), child(8, 'M')],
                 3000, 'E14', 'Flat', 3, 'Employed full-time', 5000, 'pair06'),
    make_persona('P06b', 'Couple, two young children', 1, 0,
                 [child(3, 'F'), child(5, 'F')],
                 3000, 'E14', 'Flat', 3, 'Employed full-time', 5000, 'pair06'),
))

In [11]:
# P02: couple vs lone parent with one child — both entitled to 2
pairs[1] = (
    make_persona('P02a', 'Lone parent, one child', 0, 1, [child(6, 'M')],
                 2400, 'SE13', 'Flat', 2, 'Employed full-time', 4000, 'pair02'),
    make_persona('P02b', 'Two sharers', 0, 2, [],
                 2400, 'SE13', 'Flat', 2, 'Employed full-time', 4000, 'pair02'),
)

# P05: lone parent + infant vs two sharers — both entitled to 2
pairs[4] = (
    make_persona('P05a', 'Lone parent, one infant', 0, 1, [child(2, 'M')],
                 1400, 'DA7', 'Flat', 1, 'Employed full-time', 2400, 'pair05'),
    make_persona('P05b', 'Two sharers', 0, 2, [],
                 1400, 'DA7', 'Flat', 1, 'Employed full-time', 2400, 'pair05'),
)

# P06: three children, varying only sex composition — both entitled to 3
pairs[5] = (
    make_persona('P06a', 'Couple, three children (F,F,M)', 1, 0,
                 [child(3, 'F'), child(5, 'F'), child(12, 'M')],
                 3000, 'E14', 'Flat', 3, 'Employed full-time', 5000, 'pair06'),
    make_persona('P06b', 'Couple, three children (M,M,F)', 1, 0,
                 [child(3, 'M'), child(5, 'M'), child(12, 'F')],
                 3000, 'E14', 'Flat', 3, 'Employed full-time', 5000, 'pair06'),
)

In [12]:
from sys import path
path.append('../src/agents')
from profile_agent import build_constraints

for a, b in pairs:
    ca = build_constraints(a['household'])['constraints']
    cb = build_constraints(b['household'])['constraints']
    print(f"{a['label']:28s} min_bedrooms={ca['min_bedrooms']}")
    print(f"{b['label']:28s} min_bedrooms={cb['min_bedrooms']}")
    print()

Single adult                 min_bedrooms=1
Couple                       min_bedrooms=1

Lone parent, one child       min_bedrooms=2
Two sharers                  min_bedrooms=2

Couple with two young sons   min_bedrooms=2
Couple with a son and daughter min_bedrooms=2

Lone parent, two young children min_bedrooms=2
Couple, two young children   min_bedrooms=2

Lone parent, one infant      min_bedrooms=2
Two sharers                  min_bedrooms=2

Couple, three children (F,F,M) min_bedrooms=3
Couple, three children (M,M,F) min_bedrooms=3



In [13]:
import json

# Save the six matched-entitlement pairs completed so far
with open('../data/processed/personas_partial.json', 'w') as f:
    json.dump([p for pair in pairs for p in pair], f, indent=2)

print("Saved", len([p for pair in pairs for p in pair]), "personas")

Saved 12 personas


In [14]:
check = json.load(open('../data/processed/personas_partial.json'))
print(len(check))
print([p['label'] for p in check])

12
['Single adult', 'Couple', 'Lone parent, one child', 'Two sharers', 'Couple with two young sons', 'Couple with a son and daughter', 'Lone parent, two young children', 'Couple, two young children', 'Lone parent, one infant', 'Two sharers', 'Couple, three children (F,F,M)', 'Couple, three children (M,M,F)']
